In [1]:
# import all modules needed
import numpy as np
import awkward as ak
import uproot

In [2]:
inputFilePath = '../data/example_PFNano_mcRun3_EE_allPF_noBTV.root'

print("** Open input file:", inputFilePath)
inputTree = uproot.open(inputFilePath)['Events']

N_events = inputTree.num_entries
print("** Total number of events:", N_events)

** Open input file: ../data/example_PFNano_mcRun3_EE_allPF_noBTV.root
** Total number of events: 568


List all branches in the ROOT file with `Jet_*`, `PFCands_*` or `JetPFCands_*`:

In [3]:
inputTree.show(filter_name=["Jet_*", "PFCands_*", "JetPFCands_*"])

name                 | typename                 | interpretation                
---------------------+--------------------------+-------------------------------
JetPFCands_pFCand... | int32_t[]                | AsJagged(AsDtype('>i4'))
JetPFCands_jetIdx    | int32_t[]                | AsJagged(AsDtype('>i4'))
JetPFCands_pt        | float[]                  | AsJagged(AsDtype('>f4'))
JetPFCands_dzFromPV  | float[]                  | AsJagged(AsDtype('>f4'))
JetPFCands_dxyFromPV | float[]                  | AsJagged(AsDtype('>f4'))
JetPFCands_dzErrF... | float[]                  | AsJagged(AsDtype('>f4'))
JetPFCands_dxyErr... | float[]                  | AsJagged(AsDtype('>f4'))
JetPFCands_btagEt... | float[]                  | AsJagged(AsDtype('>f4'))
JetPFCands_btagPt... | float[]                  | AsJagged(AsDtype('>f4'))
JetPFCands_btagPP... | float[]                  | AsJagged(AsDtype('>f4'))
JetPFCands_btagSi... | float[]                  | AsJagged(AsDtype('>f4'))
JetPFCands_bt

In [4]:
# define branches to be loaded from the file
inputBranchNames = ['PFCands_phi',
                     'PFCands_eta',
                     'PFCands_pt',
                     'PFCands_charge',
                     'PFCands_pdgId',
                     'PFCands_mass',
                     'PFCands_puppiWeight',
                     'JetPFCands_pFCandsIdx',
                     'JetPFCands_jetIdx',
                     'nJet',
                     'Jet_mass',
                     'Jet_eta',
                     'Jet_phi',
                     'Jet_pt',
                     'Jet_nConstituents',
                     'Jet_partonFlavour',
                     'Jet_jetId',
                     'Jet_rawFactor'
                   ]

In [5]:
# load Event bunch of 10 events
Events = inputTree.arrays(inputBranchNames, entry_start=0, entry_stop=10)   

In [6]:
# create array of particles
Particles = ak.zip({
    "PFCands_phi": Events.PFCands_phi, 
    "PFCands_eta": Events.PFCands_eta,
    "PFCands_pt": Events.PFCands_pt, 
    "PFCands_charge": Events.PFCands_charge,
    "PFCands_pdgId": Events.PFCands_pdgId, 
    "PFCands_puppiWeight": Events.PFCands_puppiWeight
})

Particles.PFCands_pt

<Array [[0.588, 0.758, 0.591, ..., 0.382, 1.34], ...] type='10 * var * float32'>

### Understanding the branches

In [7]:
# indices of the particles that belong to a jet
Events.JetPFCands_pFCandsIdx

<Array [[59, 260, 312, 313, ..., 826, 828, 817], ...] type='10 * var * int32'>

In [8]:
# indices of the jets that the particles above belong to
Events.JetPFCands_jetIdx

<Array [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], ...] type='10 * var * int32'>

In [9]:
# so, e.g., this means
e = 0

for i in range(5):
    print("Particle", Events.JetPFCands_pFCandsIdx[e][i], "belongs to jet", Events.JetPFCands_jetIdx[e][i])

Particle 59 belongs to jet 0
Particle 260 belongs to jet 0
Particle 312 belongs to jet 0
Particle 313 belongs to jet 0
Particle 314 belongs to jet 0


In [10]:
# get particles in jet j of event e
e = 0
j = 0
Particle_indices = Events.JetPFCands_pFCandsIdx[e][Events.JetPFCands_jetIdx[e] == j]
Particles[0][Particle_indices]

<Array [{PFCands_phi: 1.19, ...}, ..., {...}] type='11 * {PFCands_phi: floa...'>

In [11]:
# get particles in jet j of all events
j = 2
Particle_indices = Events.JetPFCands_pFCandsIdx[Events.JetPFCands_jetIdx == j]
Particles[Particle_indices]

<Array [[], ..., [{...}, {...}, ..., {...}]] type='10 * var * {PFCands_phi:...'>

### Extracting the particles of the jets

In [12]:
# keep only particles that are part of a jet
Particle_indices = Events.JetPFCands_pFCandsIdx
Particles = Particles[Particle_indices]
Particles

<Array [[{PFCands_phi: 1.19, ...}, ...], ...] type='10 * var * {PFCands_phi...'>

In [13]:
# flatten the events (all remaining particles become one big list)
Particles = ak.flatten(Particles)
Particles

<Array [{PFCands_phi: 1.19, ...}, ..., {...}] type='371 * {PFCands_phi: flo...'>

In [14]:
# flatten the number of particles per jet
numParticlesPerJet = ak.flatten(Events.Jet_nConstituents)
numParticlesPerJet

<Array [11, 1, 20, 2, 4, 4, 7, ..., 7, 16, 5, 2, 11, 13, 2] type='50 * uint8'>

In [15]:
# unflatten back to per-jet-level according to the number of constituents per jet
Particles = ak.unflatten(Particles, numParticlesPerJet, axis=0)
Particles

<Array [[{PFCands_phi: 1.19, ...}, ...], ...] type='50 * var * {PFCands_phi...'>

In [19]:
Particles.PFCands_phi

<Array [[1.19, 1.04, 1.13, ..., 1.22, 1.16], ...] type='50 * var * float32'>

In [20]:
Jets = ak.zip({'Jet_mass': ak.flatten(Events["Jet_mass"]),
               'Jet_eta':  ak.flatten(Events['Jet_eta']),
               'Jet_phi':  ak.flatten(Events['Jet_phi']),
               'Jet_pt': ak.flatten(Events['Jet_pt'])} |
               {k: Particles[k] for k in Particles.fields}, depth_limit=1)
Jets

<Array [{Jet_mass: 10.6, ...}, ..., {...}] type='50 * {Jet_mass: float32, J...'>

In [21]:
# print example Jet
j = 0
print("Jet", j)
print("   mass =", Jets[j].Jet_mass)
print("   pt =", Jets[j].Jet_pt)
print("   particle pt =", Jets[j].PFCands_pt)
print("   particle eta =", Jets[j].PFCands_eta)

Jet 0
   mass = 10.6484375
   pt = 120.9375
   particle pt = [0.173, 0.83, 3.66, 24.5, 7.61, 10.6, 0.693, 9.73, 3, 18.6, 21.8]
   particle eta = [-2.44, -2.87, -2.53, -2.54, -2.56, ..., -2.42, -2.52, -2.64, -2.53, -2.68]
